In [ ]:
mcpURL = "https://your-mcp-server.example.com/mcp" # Populate this variable with your remote MCP server URL

### OAuth Redirect URI

After the user authenticates with the identity provider, the IDP redirects the browser back to this URI with an authorization code in the query string (e.g. `?code=abc123&state=xyz`). We run a lightweight local HTTP server on this address to capture that code and exchange it for an access token. Some IDPs require an `https://` redirect URI — see `enable_ssl.md` for setup instructions.

When adding a 3LO remote MCP server to Quick, the IDP will need to redirect back to Quick's backend service to process the authorization code. In some cases IDP's will require allowlisting these redirects. Below is a list of redirects for Amazon Quick.

- `https://us-east-1.quicksight.aws.amazon.com/sn/oauthcallback`
- `https://us-west-2.quicksight.aws.amazon.com/sn/oauthcallback`
- `https://ap-southeast-2.quicksight.aws.amazon.com/sn/oauthcallback`
- `https://eu-west-1.quicksight.aws.amazon.com/sn/oauthcallback`
- `https://us-east-1-onebox.quicksight.aws.amazon.com/sn/oauthcallback`
- `https://us-west-2-onebox.quicksight.aws.amazon.com/sn/oauthcallback`
- `https://ap-southeast-2-onebox.quicksight.aws.amazon.com/sn/oauthcallback`
- `https://eu-west-1-onebox.quicksight.aws.amazon.com/sn/oauthcallback`



In [ ]:
# Set redirect URL, this is where we have logic that will extract the Auth Code that is in the Query String on the redirect back from the IDP
REDIRECT_URI = "http://localhost:8888/callback"

In [ ]:
from urllib.parse import urlparse as _urlparse_redirect

_parsed_redirect = _urlparse_redirect(REDIRECT_URI)
callback_host = _parsed_redirect.hostname
callback_port = _parsed_redirect.port or (443 if _parsed_redirect.scheme == 'https' else 80)

print(f"Callback host: {callback_host}")
print(f"Callback port: {callback_port}")

### Discover Protected Resource Metadata & Authorization Server

This cell runs the full MCP discovery chain:

**Phase 1 — Protected Resource Metadata (RFC 9728)**

1. HEAD the MCP endpoint and look for `WWW-Authenticate` with a `resource_metadata` URL
2. If found, fetch that URL; if not, fall back to the path-suffixed well-known URL:
   `{scheme}://{host}/.well-known/oauth-protected-resource{path}`
3. If the PRM document is valid JSON, extract `authorization_servers`, `resource`, and `scopes_supported`

**Phase 2 — Authorization Server Metadata (RFC 8414)**

4. Using the authorization server from PRM (or the MCP endpoint's origin as fallback),
   fetch `/.well-known/oauth-authorization-server{path}` then `/.well-known/openid-configuration{path}`
5. Extract `authorization_endpoint`, `token_endpoint`, and `registration_endpoint`

**Fallback behavior:** Some servers (e.g. Atlassian) skip RFC 9728 entirely but expose
RFC 8414 AS metadata at the MCP endpoint's origin. When PRM discovery fails, we try
AS metadata directly from the origin before giving up. The `resource` parameter defaults
to the MCP endpoint URL itself per RFC 8707.

In [ ]:
import requests
import re
from urllib.parse import urlparse, urljoin

def _is_json_response(resp):
    """Check that the response is both successful and has a JSON Content-Type."""
    if not resp.ok:
        return False
    content_type = resp.headers.get('Content-Type', '')
    return 'json' in content_type

def _build_wellknown(base_url, prefix):
    """Insert /.well-known/{prefix} between host and path (RFC 9728 / RFC 8414)."""
    p = urlparse(base_url)
    path = p.path.rstrip('/')
    return f"{p.scheme}://{p.netloc}/.well-known/{prefix}{path}"

# ── Phase 1: Protected Resource Metadata (RFC 9728) ──────────────────────
print("═" * 60)
print("Phase 1: Protected Resource Metadata (RFC 9728)")
print("═" * 60)

response = requests.head(mcpURL)
www_auth = response.headers.get('WWW-Authenticate', '')
print(f"\nHEAD {mcpURL}")
print(f"  Status: {response.status_code}")
print(f"  WWW-Authenticate: {www_auth or '(none)'}")

# Build list of PRM URLs to try
match = re.search(r'resource_metadata="([^"]+)"', www_auth)
prm_urls = []
if match:
    prm_urls.append(match.group(1))
fallback_prm = _build_wellknown(mcpURL, 'oauth-protected-resource')
if fallback_prm not in prm_urls:
    prm_urls.append(fallback_prm)

prm = None
for url in prm_urls:
    print(f"\nGET {url}")
    r = requests.get(url)
    ct = r.headers.get('Content-Type', '(none)')
    print(f"  Status: {r.status_code}  Content-Type: {ct}")
    if _is_json_response(r):
        prm = r.json()
        print(f"  ✓ PRM found")
        break
    print(f"  ✗ Not valid JSON PRM")

# Extract PRM fields (or set defaults for the no-PRM fallback path)
if prm:
    resource = prm.get('resource')
    authorization_servers = prm.get('authorization_servers', [])
    scopes_supported = prm.get('scopes_supported', [])
    print(f"\nProtected Resource Metadata:")
    for key, value in prm.items():
        print(f"  {key}: {value}")
else:
    # No PRM — default resource to the MCP URL itself (RFC 8707)
    resource = mcpURL
    authorization_servers = []
    scopes_supported = []
    print(f"\n⚠ PRM discovery failed — server does not implement RFC 9728.")
    print(f"  Defaulting resource to: {resource}")

# ── Phase 2: Authorization Server Metadata (RFC 8414) ────────────────────
print(f"\n{'═' * 60}")
print("Phase 2: Authorization Server Metadata (RFC 8414)")
print("═" * 60)

# Build the list of AS origins to try:
# 1. From PRM authorization_servers (if PRM succeeded)
# 2. The MCP endpoint's own origin (fallback for servers like Atlassian)
parsed_mcp = urlparse(mcpURL)
mcp_origin = f"{parsed_mcp.scheme}://{parsed_mcp.netloc}"

as_origins = list(authorization_servers)  # copy
if mcp_origin not in as_origins:
    as_origins.append(mcp_origin)

as_metadata = None
for origin in as_origins:
    for suffix in ['oauth-authorization-server', 'openid-configuration']:
        url = _build_wellknown(origin, suffix)
        print(f"\nGET {url}")
        resp = requests.get(url)
        ct = resp.headers.get('Content-Type', '(none)')
        print(f"  Status: {resp.status_code}  Content-Type: {ct}")
        if _is_json_response(resp):
            as_metadata = resp.json()
            print(f"  ✓ Found valid AS metadata")
            break
        print(f"  ✗ Skipping")
    if as_metadata:
        break

if as_metadata is None:
    raise RuntimeError(
        f"Could not fetch AS metadata from any well-known URL.\n"
        f"Tried origins: {as_origins}\n"
        f"This server may require manual endpoint configuration."
    )

print(f"\nAuthorization Server Metadata:")
for key, value in as_metadata.items():
    print(f"  {key}: {value}")

authorization_endpoint = as_metadata['authorization_endpoint']
token_endpoint = as_metadata['token_endpoint']
registration_endpoint = as_metadata.get('registration_endpoint')

print(f"\n{'═' * 60}")
print("Discovery Summary")
print("═" * 60)
print(f"  Resource:               {resource}")
print(f"  Scopes:                 {scopes_supported or '(none discovered)'}")
print(f"  Authorization Endpoint: {authorization_endpoint}")
print(f"  Token Endpoint:         {token_endpoint}")
print(f"  Registration Endpoint:  {registration_endpoint or '(none)'}")

## Step 2: Client Registration

Dynamic Client Registration (DCR, [RFC 7591](https://www.rfc-editor.org/rfc/rfc7591.html)) is **optional** per the MCP spec — servers SHOULD support it but are not required to. If the authorization server exposes a `registration_endpoint`, we attempt DCR automatically. If DCR is unavailable or fails, we fall back to prompting for a manually-obtained `client_id` and optional `client_secret` (registered via the auth server's developer dashboard).

In [ ]:
import json


client_id = None
client_secret = None

if registration_endpoint:
    # Try Dynamic Client Registration (RFC 7591)
    registration_payload = {
        "client_name": "MCP Notebook Client",
        "redirect_uris": [REDIRECT_URI],
        "grant_types": ["authorization_code", "refresh_token"],
        "response_types": ["code"],
        "token_endpoint_auth_method": "none",
        "scope": " ".join(scopes_supported) if scopes_supported else ""
    }

    print(f"Attempting DCR at: {registration_endpoint}")
    print(f"Payload: {json.dumps(registration_payload, indent=2)}")

    try:
        reg_response = requests.post(
            registration_endpoint,
            json=registration_payload,
            headers={"Content-Type": "application/json"}
        )
        reg_response.raise_for_status()
        reg_data = reg_response.json()
        client_id = reg_data['client_id']
        client_secret = reg_data.get('client_secret')
        print(f"\nDCR successful!")
        print(f"  client_id: {client_id}")
        if client_secret:
            print(f"  client_secret: {client_secret}")
    except requests.exceptions.HTTPError as e:
        print(f"\nDCR failed ({e.response.status_code}): {e.response.text}")
        print("Falling back to manual client credentials.")

if not client_id:
    # No DCR support or DCR failed — prompt for manual credentials
    print("\nDCR not available. Enter your client credentials manually.")
    print("(Register a client via the auth server's dashboard first.)")
    client_id = input("client_id: ").strip()
    client_secret = input("client_secret (leave blank for public client): ").strip() or None
    print(f"\nUsing manually provided client_id: {client_id}")

## Step 3: Authorization Code Flow with PKCE

Generate a PKCE challenge, open the browser for user consent, capture the callback on a local HTTP server, then exchange the code for an access token. The `resource` parameter (RFC 8707) binds the token to the MCP server.

In [ ]:
import hashlib
import base64
import secrets
import webbrowser
from urllib.parse import urlencode, urlparse as _urlparse, parse_qs
from http.server import HTTPServer, BaseHTTPRequestHandler
import threading

# Generate PKCE code verifier and challenge (RFC 7636)
code_verifier = secrets.token_urlsafe(64)
code_challenge = base64.urlsafe_b64encode(
    hashlib.sha256(code_verifier.encode()).digest()
).rstrip(b'=').decode()

state = secrets.token_urlsafe(32)

# Build the authorization URL
auth_params = {
    "response_type": "code",
    "client_id": client_id,
    "redirect_uri": REDIRECT_URI,
    "scope": " ".join(scopes_supported) if scopes_supported else "",
    "state": state,
    "code_challenge": code_challenge,
    "code_challenge_method": "S256",
    "resource": resource,
}

# Remove empty params to avoid confusing servers that don't expect them
auth_params = {k: v for k, v in auth_params.items() if v}

auth_url = f"{authorization_endpoint}?{urlencode(auth_params)}"
print(f"Authorization URL:\n{auth_url}\n")

# Local callback server to capture the authorization code
auth_code = None

class CallbackHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        global auth_code
        params = parse_qs(_urlparse(self.path).query)
        returned_state = params.get('state', [None])[0]
        if returned_state != state:
            self.send_response(400)
            self.end_headers()
            self.wfile.write(b'State mismatch.')
            return
        auth_code = params.get('code', [None])[0]
        self.send_response(200)
        self.send_header('Content-Type', 'text/html')
        self.end_headers()
        self.wfile.write(b'<h1>Authorization complete!</h1><p>You can close this tab.</p>')
    def log_message(self, format, *args):
        pass

server = HTTPServer((callback_host, callback_port), CallbackHandler)
server_thread = threading.Thread(target=server.handle_request)
server_thread.start()

print("Opening browser for authorization...")
webbrowser.open(auth_url)

server_thread.join(timeout=120)
server.server_close()

if auth_code:
    print(f"Authorization code received!")
else:
    print("No authorization code received (timed out or error).")

### Exchange the authorization code for an access token

In [ ]:
# Exchange authorization code for access token
token_payload = {
    "grant_type": "authorization_code",
    "code": auth_code,
    "redirect_uri": REDIRECT_URI,
    "client_id": client_id,
    "code_verifier": code_verifier,
    "resource": resource,
}

if client_secret:
    token_payload["client_secret"] = client_secret

token_response = requests.post(
    token_endpoint,
    data=token_payload,
    headers={"Content-Type": "application/x-www-form-urlencoded"}
)
token_response.raise_for_status()

token_data = token_response.json()
access_token = token_data['access_token']
refresh_token = token_data.get('refresh_token')
expires_in = token_data.get('expires_in')
token_type = token_data.get('token_type', 'Bearer')

print(f"Token type: {token_type}")
print(f"Access token: {access_token[:20]}...")
print(f"Expires in: {expires_in}s")
if refresh_token:
    print(f"Refresh token: {refresh_token[:20]}...")

print(f"\nYou can now make authenticated requests to {mcpURL} with:")
print(f"  Authorization: {token_type} <access_token>")

### Test Connection to Remote MCP server with an `initialize`

In [ ]:
headers = {
    "Authorization": f"{token_type} {access_token}",
    "Content-Type": "application/json"
}

# MCP initialize request (JSON-RPC)
init_payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {
        "protocolVersion": "2025-03-26",
        "capabilities": {},
        "clientInfo": {
            "name": "mcp-notebook-3lo",
            "version": "0.1.0"
        }
    }
}

print(f"POST {mcpURL}")
mcp_response = requests.post(mcpURL, json=init_payload, headers=headers)
print(f"  Status: {mcp_response.status_code}")
print(f"  Content-Type: {mcp_response.headers.get('Content-Type', '(none)')}")

if mcp_response.ok:
    import json
    print(f"\nResponse:\n{json.dumps(mcp_response.json(), indent=2)}")
else:
    print(f"\nError: {mcp_response.text}")

### List Available Tools

Send `tools/list` (JSON-RPC) to retrieve all tools the MCP server exposes,
along with their descriptions and input schemas.

In [ ]:
import json

tools_payload = {
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/list",
    "params": {}
}

print(f"POST {mcpURL}")
tools_response = requests.post(mcpURL, json=tools_payload, headers=headers)
print(f"  Status: {tools_response.status_code}\n")

tools_data = tools_response.json()

if 'error' in tools_data:
    print(f"Error: {tools_data['error']}")
else:
    tools = tools_data.get('result', {}).get('tools', [])
    print(f"Found {len(tools)} tool(s):\n")
    for tool in tools:
        name = tool.get('name', '(unnamed)')
        desc = tool.get('description', '(no description)')
        print(f"  ▸ {name}")
        print(f"    {desc}")
        schema = tool.get('inputSchema')
        if schema:
            props = schema.get('properties', {})
            required = schema.get('required', [])
            if props:
                print(f"    Parameters:")
                for pname, pdef in props.items():
                    req = ' (required)' if pname in required else ''
                    ptype = pdef.get('type', '?')
                    pdesc = pdef.get('description', '')
                    print(f"      - {pname}: {ptype}{req} — {pdesc}")
        print()